# 07 - Post-Training Evaluation (gold test split)

Stage P. Evaluates the newest (or pinned) model run on the **test split** -
which contains the gold human-clicked sequences that were never trained on:

1. detection metrics (ultralytics val on split=test),
2. **low/high binary accuracy** on matched detections - the metric that ties
   back to the USS use case (`docs/low_object_detection_plan.md`),
3. prediction-vs-GT galleries.

Outputs to `reports/<run_id>/`. **GPU recommended.**

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json
import random
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from vision_uss_research.datasets.yolo_export import materialize_yolo_dataset
from vision_uss_research.qa.flags import box_iou
from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run

MODEL_RUN = None       # run id under models/; None = newest completed
EVAL_TAG = "test"      # distinct report run id for holdout rotations, e.g.
                       # "holdbollard"; keep "test" for the baseline
CONF_THRESHOLD = 0.25
IOU_MATCH = 0.5
N_GALLERY = 12
SEED = 0

# class name -> low/high bin (from Label V2 semantics; see
# configs/datasets/label_v2_class_mapping.yaml)
CLASS_BINS = {"curbstone": "low", "speedbump": "low", "woodenboard": "low",
              "pole": "high", "dummychild": "high", "bicyclestand": "high",
              "low_other": "low", "high_other": "high"}
# deployment-facing category names per predicted class
CATEGORY_NAMES = {"low_other": "Low Unknown", "high_other": "High Unknown"}
def category_of(class_name: str) -> str:
    return CATEGORY_NAMES.get(class_name, CLASS_BINS.get(class_name, "?").capitalize())

model_dir = resolve_run(ARTIFACTS / "models", "train", run_id=MODEL_RUN,
                        require_file="train/weights/best.pt")
assert model_dir is not None, "no trained model run found - run notebook 06 first"
model_run = _json.loads((model_dir / "run.json").read_text())
dataset_dir = ARTIFACTS / Path(model_run["inputs"][0])

# Evaluate off LOCAL disk (Drive reads are slow and flaky). A manifest
# dataset is materialized to local (resumes 06's staging instantly);
# a legacy image dataset is read in place.
manifest = dataset_dir / "dataset_manifest.csv"
if manifest.exists():
    prov = _json.loads((dataset_dir / "provenance.json").read_text())
    eval_dir = (Path("/content/dataset") / dataset_dir.name if IN_COLAB
                else ARTIFACTS / "datasets_local" / dataset_dir.name)
    materialize_yolo_dataset(
        manifest, {k: Path(v) for k, v in prov["frames_roots"].items()},
        eval_dir, class_names=prov["class_names"], num_workers=48,
        heartbeat_file=dataset_dir / "progress.json")
else:
    eval_dir = dataset_dir
assert (eval_dir / "dataset.yaml").exists(), eval_dir

RUN_ID = new_run_id("eval", EVAL_TAG)
REPORT_DIR = ARTIFACTS / "reports" / RUN_ID
start_run(REPORT_DIR, config={"model_run": model_dir.name,
                              "conf": CONF_THRESHOLD, "iou": IOU_MATCH},
          inputs=[f"models/{model_dir.name}", f"datasets/{dataset_dir.name}"],
          repo_root=REPO_DIR)
print(f"model  : {model_dir.name}\ndataset: {dataset_dir.name}\nreport : {REPORT_DIR}")

In [ ]:
from ultralytics import YOLO

model = YOLO(str(model_dir / "train" / "weights" / "best.pt"))
class_names = yaml.safe_load((eval_dir / "dataset.yaml").read_text())["names"]

test_metrics = model.val(data=str(eval_dir / "dataset.yaml"), split="test")
detection_summary = {
    "test_mAP50": round(float(test_metrics.box.map50), 4),
    "test_mAP50_95": round(float(test_metrics.box.map), 4),
    "per_class_mAP50": {name: round(float(m), 4) for name, m in
                        zip(test_metrics.names.values(), test_metrics.box.ap50)},
}
print(_json.dumps(detection_summary, indent=1))

## Low/high binary evaluation

Greedy IoU matching of predictions to ground truth per test image, then:
per-class recall/precision at the working confidence, and the **low/high
confusion** on matched detections - a wrong bin on a matched object is the
error that matters for the USS-assist use case.

In [ ]:
def label_for(img_path):
    # ultralytics convention: images/.../x.jpg -> labels/.../x.txt
    # (works for both flat legacy dirs and per-sequence sharded dirs)
    return Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")

def read_yolo_labels(txt_path, w, h):
    boxes = []
    if txt_path.exists():
        for line in txt_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
            boxes.append((cls, (cx - bw / 2) * w, (cy - bh / 2) * h,
                          (cx + bw / 2) * w, (cy + bh / 2) * h))
    return boxes

test_images = sorted((eval_dir / "images" / "test").rglob("*.jpg"))
print(f"test images: {len(test_images)}")

counts = defaultdict(lambda: {"tp": 0, "fn": 0, "fp": 0})
bin_confusion = defaultdict(int)   # (gt_bin, pred_bin) -> n
match_rows = []

for img_path in test_images:
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    gt = read_yolo_labels(label_for(img_path), w, h)
    pred = model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
    preds = [(int(c), *map(float, b)) for c, b in
             zip(pred.boxes.cls.tolist(), pred.boxes.xyxy.tolist())]

    used = set()
    for g_cls, gx0, gy0, gx1, gy1 in gt:
        best_iou, best_j = 0.0, None
        for j, (p_cls, px0, py0, px1, py1) in enumerate(preds):
            if j in used:
                continue
            iou = box_iou((gx0, gy0, gx1, gy1), (px0, py0, px1, py1))
            if iou > best_iou:
                best_iou, best_j = iou, j
        gt_name = class_names[g_cls]
        if best_j is not None and best_iou >= IOU_MATCH:
            used.add(best_j)
            pred_name = class_names[preds[best_j][0]]
            counts[gt_name]["tp"] += 1
            bin_confusion[(CLASS_BINS.get(gt_name, "?"),
                           CLASS_BINS.get(pred_name, "?"))] += 1
            match_rows.append({"image": img_path.stem, "gt": gt_name,
                               "pred": pred_name, "iou": round(best_iou, 3)})
        else:
            counts[gt_name]["fn"] += 1
    for j, (p_cls, *_) in enumerate(preds):
        if j not in used:
            counts[class_names[p_cls]]["fp"] += 1

per_class = pd.DataFrame(counts).T
per_class["recall"] = (per_class["tp"] / (per_class["tp"] + per_class["fn"])).round(3)
per_class["precision"] = (per_class["tp"]
                          / (per_class["tp"] + per_class["fp"]).clip(lower=1)).round(3)
display(per_class)

total_matched = sum(bin_confusion.values())
bin_correct = sum(n for (g, p), n in bin_confusion.items() if g == p)
lowhigh_accuracy = round(bin_correct / max(total_matched, 1), 4)
print(f"\nlow/high binary accuracy on matched detections: {lowhigh_accuracy} "
      f"({bin_correct}/{total_matched})")
print("bin confusion:", dict(bin_confusion))

## Prediction galleries (GT green, predictions red)

In [ ]:
rng = random.Random(SEED)
sample = rng.sample(test_images, min(N_GALLERY, len(test_images)))
cols = 3
rows_n = (len(sample) + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(6.5 * cols, 4.2 * rows_n), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for cls, x0, y0, x1, y1 in read_yolo_labels(label_for(img_path), w, h):
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 2)
        cv2.putText(img, class_names[cls], (int(x0), max(int(y0) - 6, 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    pred = model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
    for cls, box in zip(pred.boxes.cls.tolist(), pred.boxes.xyxy.tolist()):
        x0, y0, x1, y1 = map(int, box)
        cv2.rectangle(img, (x0, y0), (x1, y1), (255, 40, 40), 2)
        cv2.putText(img, class_names[int(cls)], (x0, min(y1 + 18, h - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 40, 40), 2)
    ax.imshow(img)
    ax.set_title(img_path.stem, fontsize=7)
fig.savefig(REPORT_DIR / "prediction_gallery.jpg", dpi=90, bbox_inches="tight")
plt.show()

In [ ]:
per_class.to_csv(REPORT_DIR / "per_class_metrics.csv")
pd.DataFrame(match_rows).to_csv(REPORT_DIR / "matches.csv", index=False)
(REPORT_DIR / "detection_summary.json").write_text(_json.dumps(detection_summary, indent=1))

finish_run(REPORT_DIR, summary={
    **detection_summary,
    "lowhigh_accuracy": lowhigh_accuracy,
    "bin_confusion": {f"{g}->{p}": n for (g, p), n in bin_confusion.items()},
})
print(f"report: reports/{RUN_ID}")

## What next

- **Per-distance stratification**: join matches to CusReplay `ObjDist` per
  sequence to plot recall vs distance (the near-field story).
- **USS fusion**: compare/'fuse the model's low/high output with the USS
  `ClassProbHigh` on the same events - the headline experiment of
  `docs/low_object_detection_plan.md` Phase D.
- Export: `YOLO(best).export(format='onnx')` for deployment experiments.